In [4]:
"""
fig1_scatter.py
===============
Publication-ready Figure 1: r.sun modeled vs SURFRAD measured scatter plot
Three panels: Global, Beam, Diffuse radiation components

Usage:
    python fig1_scatter.py
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy import stats
from pathlib import Path

# ============================================================
# CONFIG — adjust paths
# ============================================================
BASE         = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis"
RSUN_FILE    = os.path.join(BASE, "rsun_station_values.csv")
SURFRAD_FILE = os.path.join(BASE, "surfrad_processed_0.30", "surfrad_multiyear_means.csv")
OUT_DIR      = os.path.join(BASE, "validation_outputs")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Station styling ──────────────────────────────────────────
STATION_STYLE = {
    "bon": {"color": "#1b9e77", "marker": "o",  "label": "Bondville, IL"},
    "fpk": {"color": "#d95f02", "marker": "^",  "label": "Fort Peck, MT"},
    "gwn": {"color": "#7570b3", "marker": "s",  "label": "Goodwin Creek, MS"},
    "tbl": {"color": "#e7298a", "marker": "D",  "label": "Table Mountain, CO"},
    "dra": {"color": "#66a61e", "marker": "*",  "label": "Desert Rock, NV"},
    "psu": {"color": "#e6ab02", "marker": "P",  "label": "Penn State, PA"},
    "sxf": {"color": "#a6761d", "marker": "X",  "label": "Sioux Falls, SD"},
}

COMPONENTS = ["glob_rad", "beam_rad", "diff_rad"]
PANEL_LABELS = {
    "glob_rad": "(a) Global horizontal (glob_rad)",
    "beam_rad": "(b) Direct beam (beam_rad)",
    "diff_rad": "(c) Diffuse (diff_rad)",
}

# ============================================================
# LOAD AND JOIN
# ============================================================
rsun = pd.read_csv(RSUN_FILE)
rsun = rsun[rsun["doy"] != "annual"].copy()
rsun["doy"]       = rsun["doy"].astype(int)
rsun["rsun_Whm2"] = pd.to_numeric(rsun["rsun_Whm2"], errors="coerce")
rsun = rsun[["station", "doy", "component", "rsun_Whm2"]].copy()

surfrad = pd.read_csv(SURFRAD_FILE)
surfrad["doy"] = surfrad["doy"].astype(int)
surf_long = surfrad.melt(
    id_vars=["station", "doy"],
    value_vars=["glob_rad_mean", "beam_rad_mean", "diff_rad_mean"],
    var_name="component", value_name="surfrad_Whm2"
)
surf_long["component"] = surf_long["component"].str.replace("_mean", "", regex=False)

val = rsun.merge(surf_long, on=["station", "doy", "component"], how="inner")
val = val.dropna(subset=["rsun_Whm2", "surfrad_Whm2"])
val["bias"] = val["rsun_Whm2"] - val["surfrad_Whm2"]

# ============================================================
# FIGURE
# ============================================================
plt.rcParams.update({
    "font.family"     : "Arial",
    "font.size"       : 9,
    "axes.titlesize"  : 9,
    "axes.labelsize"  : 9,
    "xtick.labelsize" : 8,
    "ytick.labelsize" : 8,
    "axes.linewidth"  : 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

fig, axes = plt.subplots(1, 3, figsize=(7.08, 2.8))  # 180mm wide — full journal page
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.22, top=0.93, wspace=0.35)

ax_min = 0

for ax, comp in zip(axes, COMPONENTS):
    sub = val[val["component"] == comp].copy()

    # Stats
    bias     = sub["bias"]
    rmse     = np.sqrt(np.mean(bias**2))
    mbe      = np.mean(bias)
    r2       = np.corrcoef(sub["rsun_Whm2"], sub["surfrad_Whm2"])[0, 1]**2
    sl, ic, *_ = stats.linregress(sub["surfrad_Whm2"], sub["rsun_Whm2"])

    ax_max = max(sub["rsun_Whm2"].max(), sub["surfrad_Whm2"].max()) * 1.08

    # 1:1 line
    ax.plot([ax_min, ax_max], [ax_min, ax_max],
            color="#999999", linestyle="--", linewidth=0.8, zorder=1, label="1:1")

    # Regression line
    xline = np.array([ax_min, ax_max])
    ax.plot(xline, sl * xline + ic,
            color="black", linewidth=0.9, zorder=2, label="Regression")

    # Data points — per station
    for st, sty in STATION_STYLE.items():
        s = sub[sub["station"] == st]
        if len(s) == 0:
            continue
        ms = 6 if sty["marker"] == "*" else 4.5
        ax.scatter(s["surfrad_Whm2"], s["rsun_Whm2"],
                   color=sty["color"], marker=sty["marker"],
                   s=ms**2, alpha=0.9, linewidths=0.3,
                   edgecolors="white", zorder=3)

    # Stats annotation — upper left
    ann = (f"$R^2$ = {r2:.3f}\n"
           f"RMSE = {rmse:.0f} Wh m$^{{-2}}$ d$^{{-1}}$\n"
           f"MBE = +{mbe:.0f} Wh m$^{{-2}}$ d$^{{-1}}$")
    ax.text(0.97, 0.04, ann,
            transform=ax.transAxes,
            fontsize=6, va="bottom", ha="right", linespacing=1.5,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                      edgecolor="#cccccc", alpha=0.9))

    ax.set_xlim(ax_min, ax_max)
    ax.set_ylim(ax_min, ax_max)
    ax.set_aspect("equal")
    ax.set_title(PANEL_LABELS[comp], fontsize=8.5, fontweight="bold", pad=4)
    ax.set_xlabel("SURFRAD measured (Wh m$^{-2}$ day$^{-1}$)", fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel("r.sun modeled (Wh m$^{-2}$ day$^{-1}$)", fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))
    ax.tick_params(direction="in", length=3)
    ax.grid(True, alpha=0.25, linewidth=0.5)

# ── Shared legend below panels ───────────────────────────────
legend_handles = [
    Line2D([0], [0], color="#999999", linestyle="--", linewidth=0.8, label="1:1 line"),
    Line2D([0], [0], color="black",   linestyle="-",  linewidth=0.9, label="Regression"),
] + [
    Line2D([0], [0], marker=sty["marker"], color=sty["color"],
           markersize=5 if sty["marker"]=="*" else 4,
           linestyle="None", label=sty["label"])
    for sty in STATION_STYLE.values()
]

fig.legend(handles=legend_handles, loc="lower center",
           ncol=5, fontsize=7.5, frameon=True,
           edgecolor="#cccccc", columnspacing=0.8, handletextpad=0.4,
           bbox_to_anchor=(0.5, -0.01))

out = os.path.join(OUT_DIR, "Fig1_scatter_pub.png")
plt.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print(f"Saved: {out}")


Saved: C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\validation_outputs\Fig1_scatter_pub.png
